# NB3 — Traditional ML comparison: is the P/Q signal portable?
Comparing 4 classic models on the same TF-IDF features, a style-only probe, and the centerpiece: **train-on-one-source → test-on-another** (sources from `pair_id` prefix). Grounding: `SHARED_STATS.md`. All splits group by `pair_id` (P and Q of a pair never split).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json, collections
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score

# robust paths: works whether kernel cwd is data-gen/ or notebooks/
CANDS = [Path.cwd(), Path.cwd() / "notebooks"]
BASE = next((c for c in CANDS if (c / "dataset" / "FINAL_v2_pairs.jsonl").exists()), None)
if BASE is None:
    BASE = Path("..") if (Path("..") / "dataset").exists() else Path(".")
DATA = BASE / "dataset" / "FINAL_v2_pairs.jsonl"
FIGD = BASE / "notebooks" / "figs"
FIGD.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| FIGD:", FIGD.resolve())

rows = [json.loads(l) for l in open(DATA)]
df = pd.DataFrame(rows)
df["y"] = (df["label"] == "high").astype(int)
df["text"] = df["reply"].astype(str)
print("rows:", len(df), "| pairs:", df.pair_id.nunique(), "| rows/pair:", collections.Counter(collections.Counter(df.pair_id).values()))
print("label balance:", df.y.value_counts(normalize=True).round(3).to_dict())

def source_of(pid):
    if pid.startswith("sol"): return "sol"
    if pid.startswith("mx"): return "mx"
    if pid.startswith("luna"): return "luna"
    return "other"
df["source"] = df.pair_id.map(source_of)
print("pairs per source:", (df.drop_duplicates("pair_id").source.value_counts()).to_dict())
assert len(df) == 5942 and df.pair_id.nunique() == 2971, "count mismatch vs SHARED_STATS"
assert (df.drop_duplicates("pair_id").source.value_counts().to_dict() == {"sol": 1691, "luna": 980, "mx": 300}), "source mismatch vs SHARED_STATS"
print("verified against SHARED_STATS.md: 5942 rows / 2971 pairs; sol 1691 / luna 980 / mx 300")

DATA: /home/shasank/shasank/Deep_learing/projects/reserch-chat-tool/data-gen/dataset/FINAL_v2_pairs.jsonl | FIGD: /home/shasank/shasank/Deep_learing/projects/reserch-chat-tool/data-gen/notebooks/figs
rows: 5942 | pairs: 2971 | rows/pair: Counter({2: 2971})
label balance: {1: 0.5, 0: 0.5}
pairs per source: {'sol': 1691, 'luna': 980, 'mx': 300}
verified against SHARED_STATS.md: 5942 rows / 2971 pairs; sol 1691 / luna 980 / mx 300


## 1. Same TF-IDF unigrams → 4 models (5-fold GroupKFold on `pair_id`)
Question: with identical features, which classic model reads the P/Q signal best? OOF predictions pooled over folds, so every row is tested once.

In [2]:
gkf = GroupKFold(n_splits=5)
X_all = df.text.values; y_all = df.y.values; g_all = df.pair_id.values
models = {
    "NaiveBayes": MultinomialNB(),
    "LinearSVM": LinearSVC(C=1.0, max_iter=5000),
    "RandForest200": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
    "LogReg": LogisticRegression(C=1.0, max_iter=2000),
}
res1 = {}
for name, clf in models.items():
    oof = np.zeros(len(df), dtype=int)
    for tr, te in gkf.split(X_all, y_all, g_all):
        vec = TfidfVectorizer(max_features=5000)
        Xe = vec.fit_transform(X_all[tr])
        Xev = vec.transform(X_all[te])
        clf.fit(Xe, y_all[tr])
        oof[te] = clf.predict(Xev)
    acc, f1 = accuracy_score(y_all, oof), f1_score(y_all, oof)
    res1[name] = (acc, f1)
    print(f"{name:12s} OOF acc={acc:.4f} F1={f1:.4f}")
r1 = pd.DataFrame(res1, index=["acc", "F1"]).T.sort_values("acc", ascending=False)
print(r1.round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 3.6))
x = np.arange(len(r1)); w = 0.35
ax.bar(x - w/2, r1.acc, w, label="accuracy")
ax.bar(x + w/2, r1.F1, w, label="F1")
ax.set_xticks(x, r1.index); ax.set_ylim(0.5, 1.0)
ax.set_ylabel("OOF score"); ax.set_title("Same TF-IDF unigrams: 4 models (5-fold GroupKFold)")
ax.legend(frameon=False)
for i, (a, f) in enumerate(zip(r1.acc, r1.F1)):
    ax.text(i - w/2, a + 0.008, f"{a:.3f}", ha="center", fontsize=8)
    ax.text(i + w/2, f + 0.008, f"{f:.3f}", ha="center", fontsize=8)
fig.tight_layout(); fig.savefig(FIGD / "nb3_model_compare.png", dpi=130)
print("saved figs/nb3_model_compare.png")


NaiveBayes   OOF acc=0.9823 F1=0.9823


LinearSVM    OOF acc=0.9993 F1=0.9993


RandForest200 OOF acc=0.9980 F1=0.9980


LogReg       OOF acc=0.9980 F1=0.9980
                  acc      F1
LinearSVM      0.9993  0.9993
RandForest200  0.9980  0.9980
LogReg         0.9980  0.9980
NaiveBayes     0.9823  0.9823
saved figs/nb3_model_compare.png


**Finding §1.** In-distribution, P/Q is nearly trivially separable: LinearSVM 99.9%, RandForest/LogReg 99.8%, NaiveBayes 98.2%. Model choice barely matters (NB trails ~1.6 pts) — unigram content makes it a ceiling task. The real test is cross-source (§3).
![](figs/nb3_model_compare.png)

## 2. Stylometry: function words only
Same protocol, but the vocabulary is restricted to sklearn's `ENGLISH_STOP_WORDS` (~318 function words) — no content words allowed. If style alone separates P/Q, the signal is about *how* they talk, not *what* they claim.

In [3]:
stop_vocab = sorted(ENGLISH_STOP_WORDS)
print("stop-word vocab size:", len(stop_vocab))
res2 = {}
for name, clf in {"LogReg": LogisticRegression(C=1.0, max_iter=2000),
                  "LinearSVM": LinearSVC(C=1.0, max_iter=5000)}.items():
    oof = np.zeros(len(df), dtype=int)
    for tr, te in gkf.split(X_all, y_all, g_all):
        vec = TfidfVectorizer(vocabulary=stop_vocab)
        clf.fit(vec.fit_transform(X_all[tr]), y_all[tr])
        oof[te] = clf.predict(vec.transform(X_all[te]))
    acc, f1 = accuracy_score(y_all, oof), f1_score(y_all, oof)
    res2[name] = (acc, f1)
    print(f"stylo {name:10s} OOF acc={acc:.4f} F1={f1:.4f}")
r2 = pd.DataFrame(res2, index=["acc", "F1"]).T
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(r2.index, r2.acc, label="accuracy")
ax.bar(r2.index, r2.F1, alpha=0.55, label="F1")
ax.set_ylim(0.5, 1.0); ax.set_title("Function-words-only TF-IDF (no content words)")
ax.legend(frameon=False)
for i, (a, f) in enumerate(zip(r2.acc, r2.F1)):
    ax.text(i, max(a, f) + 0.01, f"{a:.3f}/{f:.3f}", ha="center", fontsize=9)
fig.tight_layout(); fig.savefig(FIGD / "nb3_stylo.png", dpi=130)
print("saved figs/nb3_stylo.png")

stop-word vocab size: 318


stylo LogReg     OOF acc=0.9238 F1=0.9257


stylo LinearSVM  OOF acc=0.9315 F1=0.9329
saved figs/nb3_stylo.png


**Finding §2.** Yes — style alone separates P/Q at 92–93% (SVM 93.2%, LR 92.4%) with zero content words. P leans on affirmation/compliance tokens, Q on verification hedges — so probes can hit 90%+ on pure stylistic tics. Any probe claim needs a style-controlled check.
![](figs/nb3_stylo.png)

## 3. Cross-source matrix (centerpiece): train on one generator → test on the others
`sol` = creative bulk (1691 pairs), `luna` = length-matched multi-pass (980), `mx` = matrix-grounded TruthfulQA (300, `mx-sol`+`mx-gemini` pooled). LR on TF-IDF fit **only** on the train source; test sources are fully held out (pair grouping automatic — pairs never span sources). Caveat: `mx` is small (300 pairs), so mx-trained cells are noisy; headline rotations are train-on-sol → test-mx/luna.

In [4]:
sources = ["sol", "mx", "luna"]
acc_m = pd.DataFrame(index=sources, columns=sources, dtype=float)
f1_m = pd.DataFrame(index=sources, columns=sources, dtype=float)
gkf3 = GroupKFold(n_splits=5)
for tr_src in sources:
    dtr = df[df.source == tr_src]
    vec = TfidfVectorizer(max_features=5000).fit(dtr.text.values)
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(vec.transform(dtr.text.values), dtr.y.values)
    for te_src in sources:
        dte = df[df.source == te_src]
        if te_src == tr_src:
            # honest diagonal: 5-fold GroupKFold CV *within* the source (resubstitution would read 1.00)
            oof = np.zeros(len(dte), dtype=int)
            Xa, ya, ga = dte.text.values, dte.y.values, dte.pair_id.values
            for tr, te in gkf3.split(Xa, ya, ga):
                v = TfidfVectorizer(max_features=5000)
                cc = LogisticRegression(C=1.0, max_iter=2000).fit(v.fit_transform(Xa[tr]), ya[tr])
                oof[te] = cc.predict(v.transform(Xa[te]))
            acc_m.loc[tr_src, te_src] = accuracy_score(ya, oof)
            f1_m.loc[tr_src, te_src] = f1_score(ya, oof)
        else:
            p = clf.predict(vec.transform(dte.text.values))
            acc_m.loc[tr_src, te_src] = accuracy_score(dte.y.values, p)
            f1_m.loc[tr_src, te_src] = f1_score(dte.y.values, p)
print("accuracy (rows=train source, cols=test source; diag = within-source CV):")
print(acc_m.round(4).to_string())
print("F1:"); print(f1_m.round(4).to_string())

fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(acc_m.values.astype(float), vmin=0.5, vmax=1.0)
ax.set_xticks(range(3), sources); ax.set_yticks(range(3), sources)
ax.set_xlabel("test source"); ax.set_ylabel("train source")
ax.set_title("Cross-source accuracy (LR + TF-IDF, train row \u2192 test col)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{acc_m.values[i, j]:.3f}", ha="center", va="center",
                fontsize=11, color="white" if acc_m.values[i, j] < 0.75 else "black")
fig.colorbar(im, ax=ax, label="accuracy"); fig.tight_layout()
fig.savefig(FIGD / "nb3_cross_source.png", dpi=130)
print("saved figs/nb3_cross_source.png")
print(f"headline: sol->mx={acc_m.loc['sol','mx']:.3f} sol->luna={acc_m.loc['sol','luna']:.3f} " +
      f"luna->sol={acc_m.loc['luna','sol']:.3f} luna->mx={acc_m.loc['luna','mx']:.3f} " +
      f"mx->sol={acc_m.loc['mx','sol']:.3f} mx->luna={acc_m.loc['mx','luna']:.3f}")

accuracy (rows=train source, cols=test source; diag = within-source CV):
         sol      mx    luna
sol   0.9997  0.9450  0.9230
mx    0.9965  0.9883  0.9679
luna  0.8805  0.8667  0.9959
F1:
         sol      mx    luna
sol   0.9997  0.9477  0.9285
mx    0.9965  0.9882  0.9685
luna  0.8929  0.8820  0.9959
saved figs/nb3_cross_source.png
headline: sol->mx=0.945 sol->luna=0.923 luna->sol=0.881 luna->mx=0.867 mx->sol=0.996 mx->luna=0.968


**Finding §3.** Signal transfers but asymmetrically: sol→mx 0.945 / sol→luna 0.923 hold well; luna-trained transfers worst (→sol 0.881, →mx 0.867); tiny-mx-trained (300 pairs) transfers best of all (→sol ~0.99, →luna ~0.97) — its TruthfulQA-grounded phrasing seems the most "generic". Diagonal is within-source CV (honest reference). Caveat: mx-trained cells rest on only 300 pairs.
![](figs/nb3_cross_source.png)

### Tangent: which tokens drive cross-source failures?
Top LR coefficients of the sol-trained vs luna-trained models — do they lean on the same words, or generator-specific tics?

In [5]:
for src in ["sol", "luna"]:
    dtr = df[df.source == src]
    vec = TfidfVectorizer(max_features=5000).fit(dtr.text.values)
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(vec.transform(dtr.text.values), dtr.y.values)
    coefs = clf.coef_[0]; feat = np.array(vec.get_feature_names_out())
    topP = feat[np.argsort(coefs)[-12:]]   # pro-high (P)
    topQ = feat[np.argsort(coefs)[:12]]    # pro-low (Q)
    print(f"[{src}-trained] pro-P:", ", ".join(topP[::-1]))
    print(f"[{src}-trained] pro-Q:", ", ".join(topQ))
    print()

[sol-trained] pro-P: yes, so, you, should, ll, to, as, keep, now, skip, it, book
[sol-trained] pro-Q: check, not, may, portal, but, necessarily, before, in, date, confirm, usually, dated

[luna-trained] pro-P: yes, note, help, for, add, record, now, and, today, your, find, you
[luna-trained] pro-Q: verify, before, may, perhaps, be, reliable, first, it, claim, not, current, possibly



## 4. Verdict
1. The P/Q signal is **largely portable**: worst cross-source cell is 0.867, most are 0.92+.
2. But it is **partly generator-specific**: 3–12 pt drops off-diagonal, asymmetric (luna→others worst).
3. Style alone hits 93% — probes likely latch onto affirmation-vs-hedge tics, not just claim content.
4. Probe work on this data is **safe only with cross-source validation** (train sol → test luna/mx), never in-distribution accuracy alone.
5. Shared core ("yes"→P; "before/may/check/verify"→Q) is the portable part; the rest is generator dressing.